In [1]:
import pandas as pd

df = pd.read_csv('../data/GDXU_5min.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.reset_index(drop=True)

# Start with df copy to maintain same index
features = df[['timestamp', 'close']].copy()

# Add 10 historical closes
for i in range(1, 11):
    features[f'close_lag_{i}'] = df['close'].shift(i)

# Normalize all price columns by current close
price_cols = ['close'] + [f'close_lag_{i}' for i in range(1, 11)]
features[price_cols] = features[price_cols].div(features['close'], axis=0)

features

C:\Users\pablo\AppData\Local\Temp\ipykernel_43900\1409063948.py:4: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['timestamp'] = pd.to_datetime(df['timestamp'])


,timestamp,close,close_lag_1,close_lag_2,close_lag_3,close_lag_4,close_lag_5,close_lag_6,close_lag_7,close_lag_8,close_lag_9,close_lag_10
0,2020-12-10 08:30:00-05:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-12-10 09:30:00-05:00,1.0,1.115702,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-12-10 09:35:00-05:00,1.0,1.003317,1.119403,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-12-10 09:40:00-05:00,1.0,0.989336,0.992617,1.107465,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-12-10 09:45:00-05:00,1.0,0.993278,0.982685,0.985944,1.100020,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
129073,2025-12-11 17:55:00-05:00,1.0,1.009063,1.007451,1.008982,0.999651,0.995885,0.997094,1.007097,1.006534,1.012891,1.011481
129074,2025-12-11 18:15:00-05:00,1.0,0.990623,0.999601,0.998004,0.999521,0.990277,0.986547,0.987745,0.997654,0.997096,1.003393
129075,2025-12-11 18:20:00-05:00,1.0,1.001239,0.991851,1.000839,0.999241,1.000759,0.991504,0.987769,0.988968,0.998890,0.998331
129076,2025-12-11 18:35:00-05:00,1.0,1.004819,1.006065,0.996631,1.005663,1.004056,1.005583,0.996283,0.992530,0.993735,1.003704


In [2]:
# Calculate target on original df BEFORE feature engineering to ensure alignment
# Target: 1 if +10% hit before -5% in next 24h (288 5-min periods), else 0
def calc_target(idx):
    if idx >= len(df) - 288: return 0
    c = df['close'].iloc[idx]
    f = df['close'].iloc[idx+1:idx+1+288].values
    h, l = f >= c * 1.10, f <= c * 0.95
    if not h.any(): return 0
    if not l.any(): return 1
    return 1 if h.argmax() < l.argmax() else 0

# Add target to original df
df['target'] = [calc_target(i) for i in range(len(df))]

# Copy target to features using index alignment
features['target'] = df['target']

print(f"Target distribution in full df: {df['target'].value_counts().to_dict()}")
print(f"Target distribution in features: {features['target'].value_counts().to_dict()}")

features

Target distribution in full df: {0: 93132, 1: 35946}
Target distribution in features: {0: 93132, 1: 35946}


,timestamp,close,close_lag_1,close_lag_2,close_lag_3,close_lag_4,close_lag_5,close_lag_6,close_lag_7,close_lag_8,close_lag_9,close_lag_10,target
0,2020-12-10 08:30:00-05:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2020-12-10 09:30:00-05:00,1.0,1.115702,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2020-12-10 09:35:00-05:00,1.0,1.003317,1.119403,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2020-12-10 09:40:00-05:00,1.0,0.989336,0.992617,1.107465,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,2020-12-10 09:45:00-05:00,1.0,0.993278,0.982685,0.985944,1.100020,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
129073,2025-12-11 17:55:00-05:00,1.0,1.009063,1.007451,1.008982,0.999651,0.995885,0.997094,1.007097,1.006534,1.012891,1.011481,0
129074,2025-12-11 18:15:00-05:00,1.0,0.990623,0.999601,0.998004,0.999521,0.990277,0.986547,0.987745,0.997654,0.997096,1.003393,0
129075,2025-12-11 18:20:00-05:00,1.0,1.001239,0.991851,1.000839,0.999241,1.000759,0.991504,0.987769,0.988968,0.998890,0.998331,0
129076,2025-12-11 18:35:00-05:00,1.0,1.004819,1.006065,0.996631,1.005663,1.004056,1.005583,0.996283,0.992530,0.993735,1.003704,0


In [3]:
import talib as ta

# Calculate technical indicators from original df
h, l, c, v = df['high'].values, df['low'].values, df['close'].values, df['volume'].values

# Trend indicators
features['sma_10'] = ta.SMA(c, 10)
features['sma_20'] = ta.SMA(c, 20)
features['ema_10'] = ta.EMA(c, 10)
features['ema_20'] = ta.EMA(c, 20)
features['adx'] = ta.ADX(h, l, c, 14)

# Momentum indicators
features['rsi'] = ta.RSI(c, 14)
features['cci'] = ta.CCI(h, l, c, 14)
features['willr'] = ta.WILLR(h, l, c, 14)
features['mfi'] = ta.MFI(h, l, c, v, 14)
features['roc'] = ta.ROC(c, 10)

# MACD
features['macd'], features['macd_signal'], features['macd_hist'] = ta.MACD(c, 12, 26, 9)

# Bollinger Bands
features['bb_upper'], features['bb_middle'], features['bb_lower'] = ta.BBANDS(c, 20)

# Stochastic
features['stoch_k'], features['stoch_d'] = ta.STOCH(h, l, c, 14, 3, 3)

# Volatility
features['atr'] = ta.ATR(h, l, c, 14)

# Volume
features['obv'] = ta.OBV(c, v)
features['ad'] = ta.AD(h, l, c, v)

features

,timestamp,close,close_lag_1,close_lag_2,close_lag_3,close_lag_4,close_lag_5,close_lag_6,close_lag_7,close_lag_8,...,macd_signal,macd_hist,bb_upper,bb_middle,bb_lower,stoch_k,stoch_d,atr,obv,ad
0,2020-12-10 08:30:00-05:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.000000e+00
1,2020-12-10 09:30:00-05:00,1.0,1.115702,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-46.0,0.000000e+00
2,2020-12-10 09:35:00-05:00,1.0,1.003317,1.119403,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-96.0,5.000000e+01
3,2020-12-10 09:40:00-05:00,1.0,0.989336,0.992617,1.107465,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,159.0,2.540000e+02
4,2020-12-10 09:45:00-05:00,1.0,0.993278,0.982685,0.985944,1.100020,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,219.0,3.011429e+02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129073,2025-12-11 17:55:00-05:00,1.0,1.009063,1.007451,1.008982,0.999651,0.995885,0.997094,1.007097,1.006534,...,-0.958683,-0.081637,254.412092,250.63752,246.862948,35.596658,51.092300,1.339158,-10926318.0,-1.603731e+06
129074,2025-12-11 18:15:00-05:00,1.0,0.990623,0.999601,0.998004,0.999521,0.990277,0.986547,0.987745,0.997654,...,-0.947061,0.046487,254.173038,250.52502,246.877002,68.251044,56.326215,1.411283,-10924703.0,-1.602116e+06
129075,2025-12-11 18:20:00-05:00,1.0,1.001239,0.991851,1.000839,0.999241,1.000759,0.991504,0.987769,0.988968,...,-0.918759,0.113207,253.746952,250.36027,246.973588,73.806856,59.218186,1.332620,-10924958.0,-1.602371e+06
129076,2025-12-11 18:35:00-05:00,1.0,1.004819,1.006065,0.996631,1.005663,1.004056,1.005583,0.996283,0.992530,...,-0.898538,0.080886,253.077001,250.10602,247.135039,53.545043,65.200981,1.323147,-10925639.0,-1.602371e+06


In [4]:
# Add lagged features for all technical indicators (normalized by current value)
indicators = ['sma_10', 'sma_20', 'ema_10', 'ema_20', 'adx', 'rsi', 'cci', 'willr', 
              'mfi', 'roc', 'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_middle', 
              'bb_lower', 'stoch_k', 'stoch_d', 'atr', 'obv', 'ad']

for ind in indicators:
    for lag in range(1, 10):
        features[f'{ind}_lag_{lag}'] = features[ind].shift(lag) / features[ind]

# Drop all NaN rows
features = features.dropna()

# CRITICAL: Remove last 288 rows (24h) where target cannot be properly calculated
# These rows have insufficient future data for target calculation
original_len = len(features)
features = features.iloc[:-288]
print(f"Removed last 288 rows (24h) with insufficient future data: {original_len} -> {len(features)} rows")

features

C:\Users\pablo\AppData\Local\Temp\ipykernel_43900\1596020267.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f'{ind}_lag_{lag}'] = features[ind].shift(lag) / features[ind]
C:\Users\pablo\AppData\Local\Temp\ipykernel_43900\1596020267.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f'{ind}_lag_{lag}'] = features[ind].shift(lag) / features[ind]
C:\Users\pablo\AppData\Local\Temp\ipykernel_43900\1596020267.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame

Removed last 288 rows (24h) with insufficient future data: 122611 -> 122323 rows


,timestamp,close,close_lag_1,close_lag_2,close_lag_3,close_lag_4,close_lag_5,close_lag_6,close_lag_7,close_lag_8,...,obv_lag_9,ad_lag_1,ad_lag_2,ad_lag_3,ad_lag_4,ad_lag_5,ad_lag_6,ad_lag_7,ad_lag_8,ad_lag_9
42,2020-12-10 15:05:00-05:00,1.0,1.000215,1.004505,1.002789,1.002789,0.992062,0.999785,0.999785,0.993349,...,-1.520325,1.000000,1.000000,0.471537,0.471537,0.471537,0.471537,0.471537,0.471537,0.471537
43,2020-12-10 15:30:00-05:00,1.0,0.992758,0.992971,0.997231,0.995527,0.995527,0.984878,0.992545,0.992545,...,-0.399549,0.709657,0.709657,0.709657,0.334630,0.334630,0.334630,0.334630,0.334630,0.334630
44,2020-12-10 15:40:00-05:00,1.0,0.996181,0.988967,0.989179,0.993422,0.991725,0.991725,0.981116,0.988755,...,-0.346578,1.000000,0.709657,0.709657,0.709657,0.334630,0.334630,0.334630,0.334630,0.334630
45,2020-12-10 15:45:00-05:00,1.0,1.004304,1.000469,0.993224,0.993437,0.997699,0.995994,0.995994,0.985339,...,-0.564748,1.000000,1.000000,0.709657,0.709657,0.709657,0.334630,0.334630,0.334630,0.334630
46,2020-12-10 15:55:00-05:00,1.0,0.999191,1.003492,0.999659,0.992420,0.992633,0.996891,0.995188,0.995188,...,-0.518841,1.066259,1.066259,1.066259,0.756678,0.756678,0.756678,0.356802,0.356802,0.356802
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128777,2025-12-09 11:15:00-05:00,1.0,0.995727,0.992722,0.979292,0.962622,0.955103,0.959100,0.964242,0.966660,...,1.010987,0.999294,1.014763,1.029946,1.037730,1.040946,1.040475,1.038848,1.035755,1.038052
128778,2025-12-09 11:20:00-05:00,1.0,1.006475,1.002174,0.999149,0.985633,0.968855,0.961287,0.965310,0.970485,...,1.007813,0.995585,0.994882,1.010283,1.025399,1.033148,1.036350,1.035881,1.034261,1.031182
128779,2025-12-09 11:25:00-05:00,1.0,1.001610,1.008095,1.003787,1.000758,0.987219,0.970415,0.962835,0.966864,...,1.007524,1.003918,0.999486,0.998780,1.014241,1.029416,1.037196,1.040411,1.039940,1.038313
128780,2025-12-09 11:30:00-05:00,1.0,0.996508,0.998113,1.004575,1.000283,0.997264,0.983772,0.967026,0.959473,...,1.008466,1.002454,1.006382,1.001938,1.001231,1.016730,1.031942,1.039741,1.042964,1.042492


In [5]:
# IMPORTANT: Sort by timestamp first to ensure proper chronological split
features = features.sort_values('timestamp').reset_index(drop=True)

# Split into train (75%) and validation (25%) by timestamp
split_idx = int(len(features) * 0.75)
train = features.iloc[:split_idx].copy()
val = features.iloc[split_idx:].copy()
full = features.copy()

# Verify temporal split integrity
train_max_ts = pd.to_datetime(train['timestamp'], utc=True).max()
val_min_ts = pd.to_datetime(val['timestamp'], utc=True).min()
val_max_ts = pd.to_datetime(val['timestamp'], utc=True).max()

print("=== TEMPORAL SPLIT VERIFICATION ===")
print(f"Train set: {len(train)} rows")
print(f"  Oldest: {pd.to_datetime(train['timestamp'], utc=True).min()}")
print(f"  Newest: {train_max_ts}")
print(f"\nValidation set: {len(val)} rows")
print(f"  Oldest: {val_min_ts}")
print(f"  Newest: {val_max_ts}")
print(f"\nTime gap: {val_min_ts - train_max_ts}")

# Assert no temporal leakage
assert train_max_ts < val_min_ts, "ERROR: Training data contains timestamps newer than validation data!"
print("\n✓ Temporal split verified - no leakage detected")

# Format timestamp as ISO 8601 (handle timezone-aware datetimes)
train['timestamp'] = pd.to_datetime(train['timestamp'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%S')
val['timestamp'] = pd.to_datetime(val['timestamp'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%S')
full['timestamp'] = pd.to_datetime(full['timestamp'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%S')

train.to_csv('../data/ml/GDXU_5min_ml_train.csv', index=False)
val.to_csv('../data/ml/GDXU_5min_ml_val.csv', index=False)
full.to_csv('../data/ml/GDXU_5min_ml_full.csv', index=False)

print(f"\nSaved {len(train)} rows to data/ml/GDXU_5min_ml_train.csv")
print(f"Saved {len(val)} rows to data/ml/GDXU_5min_ml_val.csv")
print(f"Saved {len(full)} rows to data/ml/GDXU_5min_ml_full.csv")

=== TEMPORAL SPLIT VERIFICATION ===
Train set: 91742 rows
  Oldest: 2020-12-10 20:05:00+00:00
  Newest: 2024-11-21 19:10:00+00:00

Validation set: 30581 rows
  Oldest: 2024-11-21 19:15:00+00:00
  Newest: 2025-12-09 16:35:00+00:00

Time gap: 0 days 00:05:00

✓ Temporal split verified - no leakage detected

Saved 91742 rows to data/ml/GDXU_5min_ml_train.csv
Saved 30581 rows to data/ml/GDXU_5min_ml_val.csv
Saved 122323 rows to data/ml/GDXU_5min_ml_full.csv


In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, roc_curve, auc, roc_auc_score, precision_score
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd

print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting XGBoost training pipeline...")
print(f"[{datetime.now().strftime('%H:%M:%S')}] XGBoost version: {xgb.__version__}")

# Load data
print(f"[{datetime.now().strftime('%H:%M:%S')}] Loading training and validation data...")
train_df = pd.read_csv('../data/ml/GDXU_5min_ml_train.csv')
val_df = pd.read_csv('../data/ml/GDXU_5min_ml_val.csv')

# Separate features and target
X_train = train_df.drop(['timestamp', 'target'], axis=1)
y_train = train_df['target']
X_val = val_df.drop(['timestamp', 'target'], axis=1)
y_val = val_df['target']

# Clean data: replace inf/-inf with NaN, then fill with 0
print(f"[{datetime.now().strftime('%H:%M:%S')}] Cleaning data (removing inf values)...")
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_val = X_val.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Dataset info:")
print(f"  Training samples: {len(X_train):,}, Features: {X_train.shape[1]}")
print(f"  Validation samples: {len(X_val):,}")
print(f"  Target distribution (train): {y_train.value_counts().to_dict()}")
print(f"  Target distribution (val): {y_val.value_counts().to_dict()}")

# Reduced hyperparameter grid for faster training
param_grid = {
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2]
}

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Hyperparameter search configuration:")
print(f"  Method: RandomizedSearchCV")
print(f"  Number of iterations: 30")
print(f"  Cross-validation folds: 3")
print(f"  Metric: Precision (maximize)")
print(f"  Tree method: hist (CPU-optimized)")
print(f"  Parallel jobs: All CPU cores")
print(f"  Estimated time: 10-20 minutes")

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Starting hyperparameter optimization...")
print("-" * 80)

start_time = datetime.now()

# XGBoost with CPU (fast histogram method)
xgb_model = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    tree_method='hist',  # Fast CPU method
    n_jobs=-1  # Use all CPU cores
)

# RandomizedSearchCV
search = RandomizedSearchCV(
    xgb_model,
    param_grid,
    n_iter=30,
    scoring='precision',
    cv=3,
    verbose=2,
    n_jobs=-1,  # Parallel CV folds
    random_state=42
)

search.fit(X_train, y_train)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print("-" * 80)
print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Hyperparameter optimization completed in {duration/60:.1f} minutes")

# Best model
best_model = search.best_estimator_
best_params = search.best_params_

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Best hyperparameters found:")
for param, value in best_params.items():
    print(f"  {param}: {value}")
print(f"\nBest cross-validated precision score: {search.best_score_:.4f}")

# Predictions
print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Generating predictions on train and validation sets...")
y_train_pred = best_model.predict(X_train)
y_val_pred = best_model.predict(X_val)
y_train_proba = best_model.predict_proba(X_train)[:, 1]
y_val_proba = best_model.predict_proba(X_val)[:, 1]

# Classification reports
print(f"\n[{datetime.now().strftime('%H:%M:%S')}] === TRAINING SET METRICS ===")
print(classification_report(y_train, y_train_pred))
print(f"ROC AUC: {roc_auc_score(y_train, y_train_proba):.4f}")

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] === VALIDATION SET METRICS ===")
print(classification_report(y_val, y_val_pred))
print(f"ROC AUC: {roc_auc_score(y_val, y_val_proba):.4f}")

# ROC Curves
print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Generating ROC curves...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Training ROC
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_proba)
auc_train = auc(fpr_train, tpr_train)
ax1.plot(fpr_train, tpr_train, label=f'ROC (AUC = {auc_train:.4f})', linewidth=2)
ax1.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('Training Set ROC Curve')
ax1.legend()
ax1.grid(alpha=0.3)

# Validation ROC
fpr_val, tpr_val, _ = roc_curve(y_val, y_val_proba)
auc_val = auc(fpr_val, tpr_val)
ax2.plot(fpr_val, tpr_val, label=f'ROC (AUC = {auc_val:.4f})', linewidth=2)
ax2.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('Validation Set ROC Curve')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/ml/xgboost_roc_curves.png', dpi=150, bbox_inches='tight')
print(f"[{datetime.now().strftime('%H:%M:%S')}] Saved ROC curves to data/ml/xgboost_roc_curves.png")
plt.show()

# Save best model
best_model.save_model('../data/ml/xgboost_best_model.json')
print(f"[{datetime.now().strftime('%H:%M:%S')}] Saved best model to data/ml/xgboost_best_model.json")

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] ✓ Training pipeline completed successfully!")
print(f"Total elapsed time: {duration/60:.1f} minutes")

In [ ]:
features['target'].value_counts()

In [ ]:
features
